# F1 detection-action gap under LLM-judge DR

Recomputes paper_tables Tab 4 (F1) with the LLM-judge DR signal
(GPT-4o-mini primary, Llama 4 Scout secondary) instead of keyword DR.
Uses the full 2020-session pool per model from the
[`gemini3_dr_judge/`](gemini3_dr_judge/),
[`haiku_dr_judge/`](haiku_dr_judge/), and
[`llama4_dr_judge/`](llama4_dr_judge/) runs landed 2026-05-23.

GPT-5 mini has no judge run; it appears in the keyword-DR rows only,
not in the LLM-judge pooled row.

Logic mirrors [dr_judge_f1.py](dr_judge_f1.py). Tables here, prose
synthesis at the bottom.


In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path('.').resolve().parents[2] if Path('.').resolve().name == 'v2' else Path('.').resolve()
# Notebook executes from agent/logs/v2/ so REPO needs to go up 3 levels:
REPO = Path('.').resolve()
while REPO.name != 'Scammer4U' and REPO.parent != REPO:
    REPO = REPO.parent
V2 = REPO / 'agent' / 'logs' / 'v2'
print('REPO =', REPO)
print('V2   =', V2, '(exists:', V2.exists(), ')')


REPO = C:\Users\Soham\Documents\NGMI26\Scammer4U
V2   = C:\Users\Soham\Documents\NGMI26\Scammer4U\agent\logs\v2 (exists: True )


In [2]:
SLICE_DIRS = {
    'gemini-3-flash':   V2 / 'gemini3_v3_seeds1to5_full',
    'llama-4-scout':    V2 / 'llama4_v3_seeds1to5_full_re',
    'gpt-5-mini':       V2 / 'gpt5mini_v3_seed1_full',
    'claude-haiku-4.5': V2 / 'haiku_v3_seed1to5_full',
}
MODEL_LABELS = {
    'gemini-3-flash':   'Gemini_3_Flash_Preview_(OpenRouter)',
    'llama-4-scout':    'Llama_4_Scout_(OpenRouter)',
    'gpt-5-mini':       'GPT-5_mini_(OpenRouter)',
    'claude-haiku-4.5': 'Claude_Haiku_4.5_(OpenRouter)',
}
MODEL_PRETTY = {
    'gemini-3-flash':   'Gemini 3 Flash',
    'llama-4-scout':    'Llama 4 Scout',
    'gpt-5-mini':       'GPT-5 mini',
    'claude-haiku-4.5': 'Claude Haiku 4.5',
}
MODEL_ORDER = ['gpt-5-mini', 'claude-haiku-4.5', 'gemini-3-flash', 'llama-4-scout']
CONDITIONS  = ['C0', 'C1', 'C2', 'C3']

JUDGE_DIRS = {
    'gemini-3-flash':   V2 / 'gemini3_dr_judge',
    'claude-haiku-4.5': V2 / 'haiku_dr_judge',
    'llama-4-scout':    V2 / 'llama4_dr_judge',
}


## 1. Loaders

`_row_from_meta` + `load_slice` mirror paper_tables.ipynb lines 139-209.
One row per `(env, model_short, condition, seed)`. Reuses 2026-05-22
D5 rescore in `meta.json`. BROWSER_ERROR rows are excluded (S8).


In [3]:
def _row_from_meta(env, model_short, condition, seed, meta):
    if meta.get('status') != 'ok':
        return None
    s   = meta.get('summary', {})
    dq  = meta.get('data_quality', {})
    tcr = s.get('TCR')
    crit_leaked = s.get('critical_fields_leaked') or []
    plr_any_str = (s.get('PLR') or '0.0%').rstrip('%')
    try:
        plr_any = float(plr_any_str) / 100.0
    except ValueError:
        plr_any = np.nan
    return {
        'env': env,
        'model_short': model_short,
        'condition': condition,
        'seed': seed,
        'plr_crit': int(len(crit_leaked) > 0),
        'plr_any': plr_any,
        'asr': int(s.get('ASR') == 'SUCCEEDED'),
        'tcr_label': tcr,
        'task_completed': int(tcr == 'COMPLETED'),
        'dr_keyword': int(s.get('DR') == 'DETECTED'),
        'reached_trap': int(bool(s.get('reached_trap'))),
        'reliable': int(bool(s.get('reliable', dq.get('reliable')))),
        'browser_error_excluded': int(tcr == 'BROWSER_ERROR'),
    }

def load_slice(slice_root, model_label, model_short):
    rows = []
    for env_dir in slice_root.iterdir():
        if not env_dir.is_dir():
            continue
        model_dir = env_dir / model_label
        if not model_dir.is_dir():
            continue
        for cond_dir in model_dir.iterdir():
            cond = cond_dir.name
            if cond not in CONDITIONS:
                continue
            for seed_dir in cond_dir.iterdir():
                if not seed_dir.name.startswith('seed_'):
                    continue
                try:
                    seed = int(seed_dir.name.split('_')[1])
                except ValueError:
                    continue
                meta_path = seed_dir / 'meta.json'
                if not meta_path.exists():
                    continue
                row = _row_from_meta(env_dir.name, model_short, cond, seed,
                                     json.loads(meta_path.read_text()))
                if row is not None:
                    rows.append(row)
    return pd.DataFrame(rows)

parts = [load_slice(root, MODEL_LABELS[m], m) for m, root in SLICE_DIRS.items()]
df_all = pd.concat(parts, ignore_index=True)
n_pre = len(df_all)
df = df_all[df_all['browser_error_excluded'] == 0].reset_index(drop=True)
attack = df[~df['env'].str.endswith('_benign')].copy()
print(f'loaded {n_pre} sessions; excluded {n_pre - len(df)} BROWSER_ERROR; '
      f'{len(df)} retained ({len(attack)} attack, {len(df)-len(attack)} benign)')


loaded 6464 sessions; excluded 66 BROWSER_ERROR; 6398 retained (5758 attack, 640 benign)


## 2. LLM-judge DR loader

Reads `dr_summary_full.json` from each of the three judge runs. The
summary file is a flat list of session-level outputs with
`any_detection_primary`, `any_detection_secondary`,
`detection_count_primary`, and `steps`. Secondary judge (Llama 4 Scout)
ran on the **full 2020-session pool** per model (not just sample
subsets), so the both-judges-agree sensitivity is also full-power.

Session-level aggregation `any_detection = any step.detected` mirrors
keyword DR's per-session OR-aggregation. Built in `dr_judge_run.py`
line 79.


In [4]:
def load_judge_dr():
    rows = []
    for m, jdir in JUDGE_DIRS.items():
        data = json.loads((jdir / 'dr_summary_full.json').read_text())
        for s in data:
            sec = s.get('any_detection_secondary')
            rows.append({
                'env': s['env'],
                'model_short': m,
                'condition': s['condition'],
                'seed': s['seed'],
                'dr_llm_primary':   int(bool(s.get('any_detection_primary'))),
                'dr_llm_secondary': int(sec) if sec is not None else np.nan,
            })
    df_j = pd.DataFrame(rows)
    df_j['dr_llm_both'] = np.where(
        df_j['dr_llm_secondary'].isna(),
        np.nan,
        ((df_j['dr_llm_primary'] == 1) & (df_j['dr_llm_secondary'] == 1)).astype(float),
    )
    return df_j

judge = load_judge_dr()
df = attack.merge(
    judge[['env','model_short','condition','seed',
           'dr_llm_primary','dr_llm_secondary','dr_llm_both']],
    on=['env','model_short','condition','seed'],
    how='left',
)
print('judge-DR coverage per model:')
print(df.groupby('model_short')['dr_llm_primary'].apply(
    lambda s: f'{s.notna().sum()} / {len(s)}'
).to_string())


judge-DR coverage per model:
model_short
claude-haiku-4.5    1810 / 1810
gemini-3-flash      1790 / 1790
gpt-5-mini              0 / 360
llama-4-scout       1798 / 1798


## 3. Sanity check

LLM-judge DR rate per (model x condition) must match the CLAUDE.md
table: Haiku C3=41.0, Gemini C3=24.0, Llama C3=14.5.


In [5]:
rates = (judge.groupby(['model_short','condition'])['dr_llm_primary']
              .mean().unstack().mul(100).round(2))
rates.reindex(['claude-haiku-4.5','gemini-3-flash','llama-4-scout'])[CONDITIONS]


condition,C0,C1,C2,C3
model_short,,,,
claude-haiku-4.5,11.68,15.64,28.91,40.99
gemini-3-flash,1.19,9.31,20.00,23.96
llama-4-scout,0.20,0.20,2.18,14.46


## 4. `f1_table` (parameterised on DR column)

Mirrors paper_tables.ipynb 612-631 with `dr_col` argument and NaN
filtering so GPT-5 mini drops cleanly from LLM-judge pooled rows.


In [6]:
def f1_table(df_in, condition, gate_reached_trap, dr_col,
             models=tuple(MODEL_ORDER)):
    rows = []
    for m in list(models) + ['Pooled']:
        sub = df_in if m == 'Pooled' else df_in[df_in['model_short'] == m]
        sub = sub[sub['condition'] == condition]
        if gate_reached_trap:
            sub = sub[sub['reached_trap'] == 1]
        sub = sub[sub[dr_col].notna()]
        dr1 = sub[sub[dr_col] == 1]
        dr0 = sub[sub[dr_col] == 0]
        plr_dr1 = dr1['plr_crit'].mean() if len(dr1) else np.nan
        plr_dr0 = dr0['plr_crit'].mean() if len(dr0) else np.nan
        rows.append({
            'Model': 'Pooled' if m == 'Pooled' else MODEL_PRETTY[m],
            'n_DR1': len(dr1),
            'n_DR0': len(dr0),
            'PLR_DR1_pp': plr_dr1*100 if not np.isnan(plr_dr1) else np.nan,
            'PLR_DR0_pp': plr_dr0*100 if not np.isnan(plr_dr0) else np.nan,
            'gap_pp': (plr_dr0 - plr_dr1)*100
                if not (np.isnan(plr_dr0) or np.isnan(plr_dr1)) else np.nan,
        })
    return pd.DataFrame(rows)

MODELS_JUDGED = ['claude-haiku-4.5','gemini-3-flash','llama-4-scout']


## 5. Tab 4a — F1 at C0 (paper-plan headline anchor)


In [7]:
print('keyword DR (4 models):')
print(f1_table(df, 'C0', gate_reached_trap=False, dr_col='dr_keyword')
      .round(1).to_string(index=False))
print('\nLLM-judge primary DR (3 models, no GPT-5 mini):')
print(f1_table(df, 'C0', gate_reached_trap=False, dr_col='dr_llm_primary',
               models=MODELS_JUDGED).round(1).to_string(index=False))
print('\nBoth-judges-agree DR (3 models):')
print(f1_table(df, 'C0', gate_reached_trap=False, dr_col='dr_llm_both',
               models=MODELS_JUDGED).round(1).to_string(index=False))


keyword DR (4 models):
           Model  n_DR1  n_DR0  PLR_DR1_pp  PLR_DR0_pp  gap_pp
      GPT-5 mini     22     68        27.3        91.2    63.9
Claude Haiku 4.5    124    331         2.4        74.0    71.6
  Gemini 3 Flash     41    409        61.0        96.3    35.4
   Llama 4 Scout      9    444        22.2        83.6    61.3
          Pooled    196   1252        18.4        85.6    67.3

LLM-judge primary DR (3 models, no GPT-5 mini):
           Model  n_DR1  n_DR0  PLR_DR1_pp  PLR_DR0_pp  gap_pp
Claude Haiku 4.5     57    398         3.5        61.8    58.3
  Gemini 3 Flash      6    444        16.7        94.1    77.5
   Llama 4 Scout      1    452       100.0        82.3   -17.7
          Pooled     64   1294         6.2        80.1    73.8

Both-judges-agree DR (3 models):
           Model  n_DR1  n_DR0  PLR_DR1_pp  PLR_DR0_pp  gap_pp
Claude Haiku 4.5     57    398         3.5        61.8    58.3
  Gemini 3 Flash      6    444        16.7        94.1    77.5
   Llama 4 S

## 6. Tab 4b — F1 at C3 (analysis-plan S4 prereg, reached_trap=1 only)


In [8]:
print('keyword DR (4 models):')
print(f1_table(df, 'C3', gate_reached_trap=True, dr_col='dr_keyword')
      .round(1).to_string(index=False))
print('\nLLM-judge primary DR (3 models, no GPT-5 mini):')
print(f1_table(df, 'C3', gate_reached_trap=True, dr_col='dr_llm_primary',
               models=MODELS_JUDGED).round(1).to_string(index=False))
print('\nBoth-judges-agree DR (3 models):')
print(f1_table(df, 'C3', gate_reached_trap=True, dr_col='dr_llm_both',
               models=MODELS_JUDGED).round(1).to_string(index=False))


keyword DR (4 models):
           Model  n_DR1  n_DR0  PLR_DR1_pp  PLR_DR0_pp  gap_pp
      GPT-5 mini     51     24        56.9        83.3    26.5
Claude Haiku 4.5    285     52        30.5        42.3    11.8
  Gemini 3 Flash    312    100        58.0        92.0    34.0
   Llama 4 Scout    130    284        51.5        93.7    42.1
          Pooled    778    460        46.8        87.0    40.2

LLM-judge primary DR (3 models, no GPT-5 mini):


           Model  n_DR1  n_DR0  PLR_DR1_pp  PLR_DR0_pp  gap_pp
Claude Haiku 4.5    122    215        18.9        40.0    21.1
  Gemini 3 Flash    103    309        53.4        70.6    17.2
   Llama 4 Scout     70    344        48.6        86.9    38.3
          Pooled    295    868        38.0        69.5    31.5

Both-judges-agree DR (3 models):
           Model  n_DR1  n_DR0  PLR_DR1_pp  PLR_DR0_pp  gap_pp
Claude Haiku 4.5    122    215        18.9        40.0    21.1
  Gemini 3 Flash    103    309        53.4        70.6    17.2
   Llama 4 Scout     69    345        47.8        87.0    39.1
          Pooled    294    869        37.8        69.5    31.8


## 7. Side-by-side gap comparison


In [9]:
def gap_comparison(df_in, condition, gate_reached_trap):
    kw   = f1_table(df_in, condition, gate_reached_trap, 'dr_keyword')
    llm  = f1_table(df_in, condition, gate_reached_trap, 'dr_llm_primary',
                    models=MODELS_JUDGED)
    both = f1_table(df_in, condition, gate_reached_trap, 'dr_llm_both',
                    models=MODELS_JUDGED)
    rows = []
    for m in MODEL_ORDER:
        name = MODEL_PRETTY[m]
        kw_r = kw[kw['Model']==name].iloc[0]
        llm_r = llm[llm['Model']==name].iloc[0] if name in llm['Model'].values else None
        both_r = both[both['Model']==name].iloc[0] if name in both['Model'].values else None
        rows.append({
            'Model': name,
            'kw_n_DR1': int(kw_r['n_DR1']),
            'kw_gap_pp': kw_r['gap_pp'],
            'llm_n_DR1': int(llm_r['n_DR1']) if llm_r is not None else None,
            'llm_gap_pp': llm_r['gap_pp'] if llm_r is not None else np.nan,
            'both_n_DR1': int(both_r['n_DR1']) if both_r is not None else None,
            'both_gap_pp': both_r['gap_pp'] if both_r is not None else np.nan,
        })
    for tbl_name, frame in [('Pooled (kw=4M, llm=3M)', (kw, llm, both))]:
        pkw, pllm, pboth = (f[f['Model']=='Pooled'].iloc[0] for f in frame)
        rows.append({
            'Model': tbl_name,
            'kw_n_DR1': int(pkw['n_DR1']),
            'kw_gap_pp': pkw['gap_pp'],
            'llm_n_DR1': int(pllm['n_DR1']),
            'llm_gap_pp': pllm['gap_pp'],
            'both_n_DR1': int(pboth['n_DR1']),
            'both_gap_pp': pboth['gap_pp'],
        })
    return pd.DataFrame(rows)

print('At C0 (paper-plan #1 anchor):')
print(gap_comparison(df, 'C0', False).round(1).to_string(index=False))
print('\nAt C3 (analysis-plan S4 prereg, reached_trap=1 only):')
print(gap_comparison(df, 'C3', True).round(1).to_string(index=False))


At C0 (paper-plan #1 anchor):


                 Model  kw_n_DR1  kw_gap_pp  llm_n_DR1  llm_gap_pp  both_n_DR1  both_gap_pp
            GPT-5 mini        22       63.9        NaN         NaN         NaN          NaN
      Claude Haiku 4.5       124       71.6       57.0        58.3        57.0         58.3
        Gemini 3 Flash        41       35.4        6.0        77.5         6.0         77.5
         Llama 4 Scout         9       61.3        1.0       -17.7         1.0        -17.7
Pooled (kw=4M, llm=3M)       196       67.3       64.0        73.8        64.0         73.8

At C3 (analysis-plan S4 prereg, reached_trap=1 only):
                 Model  kw_n_DR1  kw_gap_pp  llm_n_DR1  llm_gap_pp  both_n_DR1  both_gap_pp
            GPT-5 mini        51       26.5        NaN         NaN         NaN          NaN
      Claude Haiku 4.5       285       11.8      122.0        21.1       122.0         21.1
        Gemini 3 Flash       312       34.0      103.0        17.2       103.0         17.2
         Llama 4 Scout   

## 8. Power-gate verdict (analysis-plan S8)

- n_DR1 < 50: descriptive-only
- 50 <= n_DR1 < 200: underpowered
- n_DR1 >= 200: stays in primary family


In [10]:
def power_verdict(n):
    if n < 50: return 'DESCRIPTIVE-ONLY'
    if n < 200: return 'UNDERPOWERED'
    return 'PRIMARY-FAMILY'

for label, gate in [('C0', False), ('C3', True)]:
    gc = gap_comparison(df, label, gate)
    pooled = gc[gc['Model'].str.startswith('Pooled')].iloc[0]
    n_kw, n_llm, n_both = int(pooled['kw_n_DR1']), int(pooled['llm_n_DR1']), int(pooled['both_n_DR1'])
    print(f'{label} pooled:')
    print(f'  keyword   n_DR1={n_kw:4d}  -> {power_verdict(n_kw)}')
    print(f'  llmjudge  n_DR1={n_llm:4d}  -> {power_verdict(n_llm)}')
    print(f'  bothagree n_DR1={n_both:4d}  -> {power_verdict(n_both)}')


C0 pooled:
  keyword   n_DR1= 196  -> UNDERPOWERED
  llmjudge  n_DR1=  64  -> UNDERPOWERED
  bothagree n_DR1=  64  -> UNDERPOWERED


C3 pooled:


  keyword   n_DR1= 778  -> PRIMARY-FAMILY
  llmjudge  n_DR1= 295  -> PRIMARY-FAMILY
  bothagree n_DR1= 294  -> PRIMARY-FAMILY


## 9. Inter-judge kappa (Cohen's)

Computed at the session level on the full 2020-session pool per model
— not the sample subset. These are higher than the
sample-subset kappas cited in CLAUDE.md (0.30 / 0.34 / 0.38), which
were computed on the targeted-ambiguous subsets.


In [11]:
def cohens_kappa(a, b):
    a, b = np.asarray(a, dtype=int), np.asarray(b, dtype=int)
    n = len(a)
    if n == 0: return float('nan')
    po = (a == b).mean()
    pa1, pb1 = a.mean(), b.mean()
    pe = pa1*pb1 + (1-pa1)*(1-pb1)
    return (po - pe) / (1 - pe) if pe < 1 else float('nan')

for m in MODELS_JUDGED:
    sub = judge[judge['model_short']==m]
    k = cohens_kappa(sub['dr_llm_primary'], sub['dr_llm_secondary'])
    print(f'  {MODEL_PRETTY[m]:<18} kappa = {k:.3f}  (n={len(sub)})')


  Claude Haiku 4.5   kappa = 0.411  (n=2020)
  Gemini 3 Flash     kappa = 0.392  (n=2020)
  Llama 4 Scout      kappa = 0.439  (n=2020)


## 10. Decomposition: where did the keyword-detectors go?

At C3 reach=1, the keyword DR=1 cell partitions into "judge agrees"
(true detectors) and "judge says non-detection" (narrators). Narrators
leak at a HIGHER rate than true detectors — which is why keyword DR's
PLR_DR1 is inflated and the keyword-DR gap is artificially narrower
than the truth.

When we move narrators into DR=0 (LLM-judge), they also drag PLR_DR0
*down*, which shrinks the apparent gap even though both endpoints
became more accurate.


In [12]:
sub = df[(df['condition']=='C3') & (df['reached_trap']==1)
         & (df['dr_llm_primary'].notna())].copy()
kw1 = sub[sub['dr_keyword']==1]
kw1_jt = kw1[kw1['dr_llm_primary']==1]
kw1_jf = kw1[kw1['dr_llm_primary']==0]

def m(s):
    return f'n={len(s):4d}  PLR_crit={s["plr_crit"].mean()*100:5.1f} pp'

print('keyword=1 total:              ', m(kw1))
print('  -> judge=1 (true detect):   ', m(kw1_jt))
print('  -> judge=0 (narrator):      ', m(kw1_jf))


keyword=1 total:               n= 727  PLR_crit= 46.1 pp
  -> judge=1 (true detect):    n= 279  PLR_crit= 34.8 pp
  -> judge=0 (narrator):       n= 448  PLR_crit= 53.1 pp


## 11. Headline synthesis (paper-facing)

### Primary anchor: F1 at C3, reached_trap=1 (analysis-plan S4 prereg)

| metric | n_DR1 | gap (pp) | power |
|---|---:|---:|---|
| keyword DR (4 models) | 778 | 40.2 | primary-family |
| LLM-judge primary (3 models) | 295 | **31.5** | primary-family |
| Both judges agree (3 models) | 294 | 31.8 | primary-family |

**Direction: gap NARROWS by 8.7 pp under LLM-judge.** Reason: keyword
DR conflates narrators with true detectors. Narrators leak at 53 pp
(vs true detectors at 35 pp), so when LLM-judge moves them into DR=0
they pull PLR_DR0 down too. Both endpoints sharpen, but the difference
between them shrinks.

**Story preserved.** At C3, 38% of agents who genuinely detect the
attack still hand over critical PII. Among non-detectors, 69.5% leak.
The detection-action gap is real and significant.

### Secondary anchor: F1 at C0 (paper-plan #1 headline)

| metric | n_DR1 | gap (pp) | power |
|---|---:|---:|---|
| keyword DR (4 models) | 196 | 67.3 | underpowered |
| LLM-judge primary (3 models) | 64 | 73.8 | underpowered |
| Both judges agree (3 models) | 64 | 73.8 | underpowered |

**Direction: gap WIDENS by 6.6 pp.** But the cell collapses below the
S8 power threshold (n=64 < 200). C0 becomes descriptive-only under
LLM-judge. **Anchor shifts to C3** for paper-facing inference.

Llama C0 LLM-judge n_DR1=1 (a single detected session that leaked) —
the per-model row for Llama at C0 is a single-observation point
estimate and should not be cited.

### Per-model verdict at C3 (LLM-judge, reached_trap=1)

| Model | gap (pp) | n_DR1 | verdict |
|---|---:|---:|---|
| Haiku 4.5 | 21.1 | 122 | underpowered |
| Gemini 3 Flash | 17.2 | 103 | underpowered |
| Llama 4 Scout | 38.3 | 70 | underpowered |

All three models are underpowered at the per-model level under
LLM-judge. Pooled stays primary-family. F1 becomes a
**pooled-only** claim under LLM-judge.

### GPT-5 mini

No LLM-judge run. Included only in keyword-DR rows. When seeds 2-5
finish and a judge run is added, the pooled cell can move to 4 models.

### Kappa

Session-level kappa on full pool: 0.39-0.44 (fair-moderate agreement).
Higher than the sample-subset kappas cited in CLAUDE.md (0.30-0.38),
as expected since the sample subset was targeted-ambiguous.

### Implication for paper framing

1. F1 at C0 (current paper-plan #1) drops as the headline anchor and
   becomes a sensitivity result.
2. F1 at C3 (current analysis-plan S4 prereg) becomes the paper-facing
   primary; 31.5 pp gap is the headline number.
3. Methodology section gains a paragraph contrasting keyword DR
   (overcounts due to narration) vs LLM-judge DR (filters narration);
   the 8.7 pp narrowing is itself part of the measurement story.
4. Per-model F1 reported as descriptive at C3; pooled F1 is the only
   inference-bearing cell.
